# 03 - Embedding and Delta Embedding Generation

This notebook performs:

1. Embedding extraction
2. Delta embedding generation

Face Recognition Systems:

- AdaFace
- ArcFace
- MagFace
- ElasticFace
- EdgeFace

Inputs:

- image-output/

Outputs:

- Embeddings/
- Embeddings_Diff/

Expected Runtime:

Demo Mode:
- 40min - 2hour

Full Dataset:
- several hours

GPU Required:

- Embedding Extraction: Yes
- Delta Embeddings: No

In [1]:
try:
    from google.colab import drive

    drive.mount("/content/drive")

    print("Google Drive mounted.")

except ImportError:

    print("Running outside Colab. Drive mount skipped.")

Mounted at /content/drive


In [2]:
from pathlib import Path
import os
import sys
import random
import traceback
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms


RANDOM_SEED = 0
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
torch.cuda.manual_seed_all(RANDOM_SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


## Configuration

This section defines:

- Project paths
- Repository locations
- Model checkpoints
- Output directories

In [3]:
PROJECT_ROOT = Path("/content/drive/MyDrive/Free-Cloud/ICPR_Rep")

REPOS_DIR = PROJECT_ROOT/"repos"
MODELS_DIR = PROJECT_ROOT/"models"
IMG_ROOT = PROJECT_ROOT/"image-output"
EMB_ROOT = PROJECT_ROOT/"Embeddings"
EMB_ROOT.mkdir(parents=True,exist_ok=True)

print("Image Root :",IMG_ROOT)
print("Embedding Root :",EMB_ROOT)

Image Root : /content/drive/MyDrive/Free-Cloud/ICPR_Rep/image-output
Embedding Root : /content/drive/MyDrive/Free-Cloud/ICPR_Rep/Embeddings


In [4]:
ADAFACE_REPO = REPOS_DIR / "AdaFace"
MAGFACE_REPO = REPOS_DIR / "MagFace"
ELASTICFACE_REPO = REPOS_DIR / "ElasticFace"
EDGEFACE_REPO = REPOS_DIR / "edgeface"

for repo in [ADAFACE_REPO, MAGFACE_REPO, ELASTICFACE_REPO, EDGEFACE_REPO]:
    if str(repo) not in sys.path:
        sys.path.insert(0, str(repo))

print("Repositories registered.")

Repositories registered.


In [5]:
# ArcFace Architecture

def conv3x3(in_planes, out_planes, stride=1, groups=1, dilation=1):
    return nn.Conv2d(in_planes, out_planes, kernel_size=3, stride=stride, padding=dilation, groups=groups, bias=False, dilation=dilation)


def conv1x1(in_planes, out_planes, stride=1):
    return nn.Conv2d(in_planes, out_planes, kernel_size=1, stride=stride, bias=False)


class IBasicBlock(nn.Module):
    expansion = 1

    def __init__(self, inplanes, planes, stride=1, downsample=None):
        super().__init__()
        self.bn1 = nn.BatchNorm2d(inplanes, eps=1e-5)
        self.conv1 = conv3x3(inplanes, planes)
        self.bn2 = nn.BatchNorm2d(planes, eps=1e-5)
        self.prelu = nn.PReLU(planes)
        self.conv2 = conv3x3(planes, planes, stride)
        self.bn3 = nn.BatchNorm2d(planes, eps=1e-5)
        self.downsample = downsample

    def forward(self, x):
        identity = x
        out = self.bn1(x)
        out = self.conv1(out)
        out = self.bn2(out)
        out = self.prelu(out)
        out = self.conv2(out)
        out = self.bn3(out)

        if self.downsample is not None:
            identity = self.downsample(x)

        out += identity
        return out


class IResNet_Legacy(nn.Module):

    def __init__(self, block, layers, num_features=512):
        super().__init__()
        self.inplanes = 64
        self.conv1 = nn.Conv2d(3, self.inplanes, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(self.inplanes, eps=1e-5)
        self.prelu = nn.PReLU(self.inplanes)

        self.layer1 = self._make_layer(block, 64, layers[0], stride=2)
        self.layer2 = self._make_layer(block, 128, layers[1], stride=2)
        self.layer3 = self._make_layer(block, 256, layers[2], stride=2)
        self.layer4 = self._make_layer(block, 512, layers[3], stride=2)

        self.bn2 = nn.BatchNorm2d(512, eps=1e-5)
        self.dropout = nn.Dropout(p=0.4, inplace=True)
        self.fc = nn.Linear(512 * 7 * 7, num_features)
        self.features = nn.BatchNorm1d(num_features, eps=1e-5)

    def _make_layer(self, block, planes, blocks, stride=1):
        downsample = None
        if stride != 1 or self.inplanes != planes:
            downsample = nn.Sequential(
                conv1x1(self.inplanes, planes, stride),
                nn.BatchNorm2d(planes, eps=1e-5)
            )

        layers = [block(self.inplanes, planes, stride, downsample)]
        self.inplanes = planes
        for _ in range(1, blocks):
            layers.append(block(self.inplanes, planes))

        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.prelu(x)

        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)

        x = self.bn2(x)
        x = torch.flatten(x, 1)
        x = self.dropout(x)
        x = self.fc(x)
        x = self.features(x)
        return x


def arcface_iresnet100():
    return IResNet_Legacy(IBasicBlock, [3, 13, 30, 3])

print("ArcFace architecture ready.")

ArcFace architecture ready.


In [6]:
# Adaface necessities

def load_checkpoint(ckpt_path):
    ckpt = torch.load(ckpt_path, map_location="cpu")
    if isinstance(ckpt, dict):
        if "state_dict" in ckpt: return ckpt["state_dict"]
        if "model" in ckpt: return ckpt["model"]
    return ckpt


def try_key_strips(sd):
    variants = {}
    prefixes = ["module.", "backbone.", "model."]

    for p in prefixes:
        variants[p] = {k.replace(p, ""): v for k, v in sd.items()}

    variants["original"] = sd
    return variants


def build_remapped_state_dict(model_keys, ckpt_state):
    model_keys = set(model_keys)
    remapped = {}
    variants = try_key_strips(ckpt_state)

    for variant in variants.values():
        for k, v in variant.items():
            if k in model_keys and k not in remapped:
                remapped[k] = v

    return remapped

print("AdaFace helper functions ready.")

AdaFace helper functions ready.


In [7]:
preprocess_rgb = transforms.Compose([
    transforms.Resize((112, 112)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

print("Preprocessing ready.")

def load_image_tensor(image_path):
    image = Image.open(image_path).convert("RGB")
    tensor = preprocess_rgb(image)
    return tensor.unsqueeze(0).to(device)

Preprocessing ready.


## Face Recognition Systems

The following FRS models are loaded:

- AdaFace
- ArcFace
- MagFace
- ElasticFace
- EdgeFace

All embeddings are L2-normalized and stored in NumPy format.

In [8]:
# AdaFace

from net import build_model

adaface_ckpt = MODELS_DIR / "adaface_ir101_ms1mv3.ckpt"
adaface_model = build_model("ir_101")

ckpt_state = load_checkpoint(str(adaface_ckpt))
mapped_state = build_remapped_state_dict(adaface_model.state_dict().keys(), ckpt_state)

adaface_model.load_state_dict(mapped_state, strict=False)
adaface_model = adaface_model.to(device).eval()

print("AdaFace loaded successfully.")

AdaFace loaded successfully.


In [9]:
# Load MagFace

from models.iresnet import iresnet100 as magface_iresnet100

magface_ckpt = MODELS_DIR / "magface-r100-glint360k.pth"
magface_model = magface_iresnet100()

ckpt = torch.load(magface_ckpt, map_location=device)
magface_model.load_state_dict(ckpt)
magface_model = magface_model.to(device).eval()

print("MagFace loaded successfully.")

MagFace loaded successfully.


In [10]:
# Load ArcFace

arcface_ckpt = MODELS_DIR / "arcface_r100.pth"
arcface_model = arcface_iresnet100()

arcface_state = torch.load(arcface_ckpt, map_location=device)
arcface_model.load_state_dict(arcface_state)
arcface_model = arcface_model.to(device).eval()

print("ArcFace loaded successfully.")

ArcFace loaded successfully.


In [11]:
# ElasticFace

import importlib.util

def load_elasticface_model(checkpoint_path):
    print("Loading ElasticFace...")
    iresnet_py = ELASTICFACE_REPO / "backbones" / "iresnet.py"
    if not iresnet_py.exists():
        raise FileNotFoundError(f"{iresnet_py}")

    spec = importlib.util.spec_from_file_location("elastic_iresnet_module", str(iresnet_py))
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)

    model = mod.iresnet100(pretrained=False)
    model = model.to(device)

    ckpt = torch.load(checkpoint_path, map_location="cpu")
    state_dict = ckpt

    for k in ["state_dict", "model_state", "backbone", "net", "model"]:
        if isinstance(ckpt, dict) and k in ckpt:
            state_dict = ckpt[k]
            break

    new_state = {}
    for k, v in state_dict.items():
        nk = k
        if nk.startswith("module."):
            nk = nk[len("module."):]
        if nk.startswith("backbone."):
            nk = nk[len("backbone."):]
        new_state[nk] = v

    model.load_state_dict(new_state, strict=False)
    model.eval()
    return model

elasticface_ckpt = MODELS_DIR / "elasticface_295672backbone.pth"
elasticface_model = load_elasticface_model(elasticface_ckpt)
print("ElasticFace loaded.")

Loading ElasticFace...
ElasticFace loaded.


In [12]:
# EdgeFace

import hubconf as hub

edgeface_ckpt = MODELS_DIR / "edgeface_base.pt"

edgeface_model = hub.edgeface_base(pretrained=False)
edgeface_model = edgeface_model.to(device)

ckpt = torch.load(edgeface_ckpt, map_location="cpu")
state_dict = ckpt

for k in ["state_dict", "model", "net", "model_state"]:
    if isinstance(ckpt, dict) and k in ckpt:
        state_dict = ckpt[k]
        break

try:
    edgeface_model.load_state_dict(state_dict, strict=True)
except:
    fixed_state = {}
    for k, v in state_dict.items():
        if not k.startswith("model."):
            fixed_state["model." + k] = v
        else:
            fixed_state[k] = v
    edgeface_model.load_state_dict(fixed_state, strict=True)

edgeface_model.eval()
print("EdgeFace loaded.")

EdgeFace loaded.


## Model Registry

A unified interface is created for all FRS models.

This allows embedding extraction to be performed using a common pipeline.

In [13]:
# Unified Model Registry

models = {
    "adaface": adaface_model,
    "arcface": arcface_model,
    "magface": magface_model,
    "elasticface": elasticface_model,
    "edgeface": edgeface_model
}

print("\nLoaded Models:")

for m in models:
    print(m)


Loaded Models:
adaface
arcface
magface
elasticface
edgeface


In [14]:
IMG_EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff", ".webp"}

all_images = []

for path in IMG_ROOT.rglob("*"):
    if path.is_file() and path.suffix.lower() in IMG_EXTS:
        all_images.append(path)

all_images = sorted(all_images)

print(f"Images found: {len(all_images)}")

Images found: 1754


In [15]:
def get_embedding(model, image_tensor):
    with torch.no_grad():
        embedding = model(image_tensor)

        if isinstance(embedding, tuple):
            embedding = embedding[0]

        embedding = F.normalize(embedding, p=2, dim=1)

    return embedding.squeeze(0).cpu().numpy().astype(np.float32)

In [16]:
def build_embedding_path(image_path, model_name):
    relative_path = image_path.relative_to(IMG_ROOT)
    output_path = EMB_ROOT / model_name / relative_path
    output_path = output_path.with_suffix(".npy")
    return output_path

## Embedding Extraction

Embeddings are extracted for every image contained in:

image-output/

and mirrored into:

Embeddings/

while preserving the complete directory structure.

In [18]:
# Extract Embeddings

total_saved = 0

for model_name, model in models.items():
    print("\n" + "=" * 80)
    print(f"Processing {model_name}")
    print("=" * 80)

    saved_count = 0

    for image_path in tqdm(all_images, desc=model_name):
        try:
            image_tensor = load_image_tensor(image_path)
            embedding = get_embedding(model, image_tensor)
            output_path = build_embedding_path(image_path, model_name)
            output_path.parent.mkdir(parents=True, exist_ok=True)
            np.save(output_path, embedding)
            saved_count += 1

        except Exception as e:
            print(f"\nERROR: {image_path}")
            print(e)

    print(f"{model_name}: {saved_count}")
    total_saved += saved_count

print("\n" + "=" * 80)
print(f"Total Embeddings Saved: {total_saved}")


Processing adaface


adaface:   0%|          | 0/1754 [00:00<?, ?it/s]

adaface: 1754

Processing arcface


arcface:   0%|          | 0/1754 [00:00<?, ?it/s]

arcface: 1754

Processing magface


...

magface: 1754

Processing elasticface


elasticface:   0%|          | 0/1754 [00:00<?, ?it/s]

elasticface: 1754

Processing edgeface


edgeface:   0%|          | 0/1754 [00:00<?, ?it/s]

edgeface: 1754

Total Embeddings Saved: 8770


# Part B - Delta Embeddings

Delta embeddings are generated for:

1. Genuine comparisons
2. Morph comparisons

Outputs are stored under:

Embeddings_Diff/

and are later used by:

04_dmad_evaluation.ipynb

In [2]:
from pathlib import Path
import os
import re
import glob
import logging
from collections import defaultdict
from itertools import combinations
import numpy as np
from tqdm.auto import tqdm

## Delta Configuration

This section defines:

- Dataset list
- FRS list
- Output directories
- Logging

In [3]:
# change paths as per your execution paths

PROJECT_ROOT = Path("/content/drive/MyDrive/Free-Cloud/ICPR_Rep")

EMB_ROOT      = PROJECT_ROOT / "Embeddings"
EMB_DIFF_ROOT = PROJECT_ROOT / "Embeddings_Diff"

EMB_DIFF_ROOT.mkdir(parents=True, exist_ok=True)

print("Embeddings Root:")
print(EMB_ROOT)

print("\nDelta Root:")
print(EMB_DIFF_ROOT)

Embeddings Root:
/content/drive/MyDrive/Free-Cloud/ICPR_Rep/Embeddings

Delta Root:
/content/drive/MyDrive/Free-Cloud/ICPR_Rep/Embeddings_Diff


In [4]:
DATASETS = ["FERET", "FRGC"]

MODELS = ["adaface", "arcface", "magface", "elasticface", "edgeface"]

print("Datasets:", DATASETS)
print("Models:", MODELS)

Datasets: ['FERET', 'FRGC']
Models: ['adaface', 'arcface', 'magface', 'elasticface', 'edgeface']


In [5]:
log_file_path = EMB_DIFF_ROOT / "delta_creation_log.txt"

root_logger = logging.getLogger()

if root_logger.hasHandlers():
    root_logger.handlers.clear()

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] - %(message)s",
    handlers=[
        logging.FileHandler(log_file_path, mode="w"),
        logging.StreamHandler()
    ]
)

log_file = logging.getLogger()
log_file.info("Delta Embedding Notebook Started")

2026-06-10 19:38:06,622 [INFO] - Delta Embedding Notebook Started


In [6]:
def l2_normalize(v):
    norm = np.linalg.norm(v)
    if norm == 0:
        return v
    return v / norm


def get_base_name(filepath):
    return Path(filepath).stem

## Filename Parsing

Helper functions are used to:

- Parse identities
- Parse morph filenames
- Locate source embeddings

while preserving compatibility across FERET and FRGC.

In [7]:
# Filename

def parse_bonafide_filename(filename):
    base = Path(filename).stem
    m = re.match(r"^([0-9]+)", base)
    if m:
        return m.group(1)
    return None


def parse_morph_filename(filename):
    base = Path(filename).stem

    if "-vs-" not in base:
        return None, None

    left, right = base.split("-vs-")

    s1 = re.match(r"^([0-9]+)", left)
    s2 = re.match(r"^([0-9]+)", right)

    if not s1 or not s2:
        return None, None

    return s1.group(1), s2.group(1)

In [9]:
# Anchor Index

def build_anchor_index(aligned_root):
    anchor_index = {
        "train": defaultdict(list),
        "test": defaultdict(list)
    }

    for split in ["train", "test"]:
        split_dir = aligned_root / split

        if not split_dir.exists():
            continue

        for emb_path in split_dir.glob("*.npy"):
            person_id = parse_bonafide_filename(emb_path.name)

            if person_id:
                anchor_index[split][person_id].append(emb_path)

    return anchor_index

## Genuine Delta Embeddings

For every identity:

embedding_a - embedding_b

is computed for all available bona fide image pairs.

In [10]:
def generate_genuine_deltas(anchor_index, data_root, diff_root):
    count = 0

    for split in ["train", "test"]:
        for person_id, emb_paths in anchor_index[split].items():
            if len(emb_paths) < 2:
                continue

            for path1, path2 in combinations(emb_paths, 2):
                e1 = l2_normalize(np.load(path1))
                e2 = l2_normalize(np.load(path2))
                delta = e1 - e2

                out_dir = Path(str(path1.parent).replace(str(data_root), str(diff_root)))
                out_dir.mkdir(parents=True, exist_ok=True)

                out_file = out_dir / f"{get_base_name(path1)}-vs-{get_base_name(path2)}.npy"
                np.save(out_file, delta)
                count += 1

    return count

In [11]:
def get_morph_files(morph_root):
    morph_files = {"train": [], "test": []}

    for split in ["train", "test"]:
        split_files = list(morph_root.rglob(f"{split}/*.npy"))
        morph_files[split].extend(split_files)

    return morph_files

## Morph Delta Embeddings

For every morph:

delta = morph_embedding - source_embedding

is generated and stored for downstream D-MAD evaluation.

In [12]:
def generate_imposter_deltas(anchor_index, morph_root, data_root, diff_root):
    count = 0
    morph_files = get_morph_files(morph_root)

    for split in ["train", "test"]:
        for morph_path in tqdm(morph_files[split], desc=f"Imposter {split}"):
            try:
                e_morph = l2_normalize(np.load(morph_path))
                s1_pid, s2_pid = parse_morph_filename(morph_path.name)

                if s1_pid is None or s2_pid is None:
                    continue

                s1_anchors = anchor_index[split].get(s1_pid, [])
                s2_anchors = anchor_index[split].get(s2_pid, [])
                all_anchors = s1_anchors + s2_anchors

                if len(all_anchors) == 0:
                    continue

                for anchor_path in all_anchors:
                    e_anchor = l2_normalize(np.load(anchor_path))
                    delta = e_morph - e_anchor

                    out_dir = Path(str(morph_path.parent).replace(str(data_root), str(diff_root)))
                    out_dir.mkdir(parents=True, exist_ok=True)

                    out_file = out_dir / f"{get_base_name(morph_path)}_vs_{get_base_name(anchor_path)}.npy"
                    np.save(out_file, delta)
                    count += 1

            except Exception as e:
                print(f"ERROR: {morph_path}")
                print(e)

    return count

## Delta Generation

This section creates:

- Genuine delta embeddings
- Morph delta embeddings

for all:

- Datasets
- FRS models
- Perturbation types

In [13]:
for model in tqdm(MODELS, desc="Models"):
    for dataset in tqdm(DATASETS, leave=False):
        dataset_root = EMB_ROOT / model / dataset

        if not dataset_root.exists():
            continue

        versions = [d for d in dataset_root.iterdir() if d.is_dir()]

        for version_dir in versions:
            version_name = version_dir.name

            print("\n" + "=" * 80)
            print(f"{model} | {dataset} | {version_name}")
            print("=" * 80)

            aligned_root = version_dir / "aligned"
            morph_root   = version_dir / "morph"

            if not aligned_root.exists():
                print("No aligned folder.")
                continue

            anchor_index  = build_anchor_index(aligned_root)
            genuine_count = generate_genuine_deltas(anchor_index, EMB_ROOT, EMB_DIFF_ROOT)
            print(f"Genuine Deltas: {genuine_count}")

            if morph_root.exists():
                imposter_count = generate_imposter_deltas(anchor_index, morph_root, EMB_ROOT, EMB_DIFF_ROOT)
                print(f"Imposter Deltas: {imposter_count}")
            else:
                print("No morph folder found.")

Models:   0%|          | 0/5 [00:00<?, ?it/s]

  0%|          | 0/2 [00:00<?, ?it/s]


adaface | FERET | Weighted_Ensemble_Template_BPDA_EOT
Genuine Deltas: 43


Imposter train:   0%|          | 0/57 [00:00<?, ?it/s]

Imposter test:   0%|          | 0/54 [00:00<?, ?it/s]

...

Imposter test:   0%|          | 0/60 [00:00<?, ?it/s]

Imposter Deltas: 2160

edgeface | FRGC | original
Genuine Deltas: 418


Imposter train:   0%|          | 0/60 [00:00<?, ?it/s]

Imposter test:   0%|          | 0/60 [00:00<?, ?it/s]

Imposter Deltas: 2160


## Generated Directory Structure
``` text
Embeddings/

├── adaface/
├── arcface/
├── magface/
├── elasticface/
└── edgeface/

Embeddings_Diff/

├── adaface/
├── arcface/
├── magface/
├── elasticface/
└── edgeface/
```
The directory structure mirrors image-output/ exactly.